# Análisis de experimentos federados con XGBoost (Adidas)


Este cuaderno reúne todo lo necesario para:
1. **Lanzar o relanzar experimentos** definidos en `configs/`.
2. **Cargar métricas** guardadas en `results/`.
3. **Graficar** evolución de RMSE, tiempos y **analizar la importancia de variables**.

In [ ]:
#import os
#os.system("pip install -e .")



## Limpieza de los datos

In [ ]:
import pandas as pd
from pathlib import Path


RAW_FILE = Path("../../datos/Adidas US Sales Datasets.xlsx").resolve()

def load_clean_adidas(
    path: Path | str = RAW_FILE,
    build_partition: str = "retailer_region",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Carga el Excel de Adidas, elimina cabeceras fantasma, convierte tipos y añade
    columnas útiles. Devuelve un DataFrame listo para modelado o particionado.
    
    Parameters
    ----------
    path : Path | str
        Ruta al archivo Excel.
    build_partition : {"retailer_region", "retailer", "region", None}
        Cómo construir la columna 'partition_id'.  None → no la crea.
    verbose : bool
        Si True muestra prints y heads para depuración en notebook.
    """
    # -------- 1. Leer y sanear cabecera -------------------------------------------------
    df = pd.read_excel(path)

    # Las tres primeras filas son basura; la 4.ª contiene la cabecera real
    df = df.drop(index=[0, 1, 2])
    header_row = df.loc[3]
    df = df.rename(columns=header_row).drop(index=3).reset_index(drop=True)

    # El Excel trae una primera columna vacía → fuera
    if df.columns[0] == "":
        df = df.drop(columns=df.columns[0])

    # -------- 2. Convertir tipos --------------------------------------------------------
    # A) Fechas
    df["Invoice Date"] = pd.to_datetime(df["Invoice Date"], errors="coerce")
    df = df[df["Invoice Date"].notna()]

    # B) Categóricas y numéricas explícitas
    cat_cols = [
        "Retailer", "Region", "State", "City",
        "Product", "Sales Method",
    ]
    num_cols = [
        "Price per Unit", "Total Sales",
        "Operating Profit", "Operating Margin",
        "Units Sold",
    ]
    for col in cat_cols:
        if col in df:
            df[col] = df[col].astype("category")
    for col in num_cols:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # -------- 3. Columnas derivadas -----------------------------------------------------
    if "Units Sold" in df and "Price per Unit" in df:
        df["RevenueUSD"] = df["Units Sold"] * df["Price per Unit"]

    if build_partition:
        match build_partition:
            case "retailer_region":
                df["partition_id"] = (
                    df["Retailer"].str.strip() + " - " + df["Region"].str.strip()
                )
            case "retailer":
                df["partition_id"] = df["Retailer"].str.strip()
            case "region":
                df["partition_id"] = df["Region"].str.strip()
            case _:
                raise ValueError(f"build_partition='{build_partition}' no reconocido")

    # -------- 4. Verbose diagnostics ----------------------------------------------------
    if verbose:
        print("✅ Limpieza terminada")
        display(df.head())
        print("\nTipos finales:")
        display(df.dtypes)

    return df

# --- prueba rápida ---
df = load_clean_adidas(verbose=False)
print("Filas totales tras limpieza:", len(df))


## Generar configuraciones TOML

In [ ]:
from itertools import product
from pathlib import Path
import toml, datetime

CONFIG_DIR = Path("../configs")
CONFIG_DIR.mkdir(exist_ok=True)

# Grids completos, pero quitamos central_evals
partitioners  = ["retailer_region"]
strategies    = ["bagging", "cycling"]
test_fracs    = [0.1, 0.2, 0.3]
local_epochs  = [1, 5, 10]
etas          = [0.05, 0.1]
max_depths    = [4, 6, 8]
subsamples    = [0.6, 0.8, 1.0]

generated = []
today = datetime.date.today().isoformat()

for part, strat, tf, le, eta, md, ss in product(
        partitioners, strategies,
        test_fracs, local_epochs, etas, max_depths, subsamples
):
    # Nombre incluye que siempre es CE-on
    name = (f"{part}-{strat}-ce-on-tf-{tf}-le-{le}"
            f"-eta-{eta}-md-{md}-ss-{ss}").replace(".", "p")

    cfg = {
        "run-id": name + "_" + today,
        "strategy": strat,
        "partitioner": part,
        "centralised-eval": True,      # siempre evaluación global
        "test-fraction": tf,
        "local-epochs": le,
        "params": {
            "eta": eta,
            "max_depth": md,
            "subsample": ss,
        },
    }

    toml.dump(cfg, open(CONFIG_DIR / f"{name}.toml", "w", encoding="utf-8"))
    generated.append(name)

print(f"📝 {len(generated)} configuraciones creadas en {CONFIG_DIR}")


## Ejecutar Batch de experimentos

### Un único experimento de prueba

In [ ]:
import subprocess, time
import pandas as pd
from pathlib import Path
import toml

SRC_DIR   = Path("..").resolve()  # debe contener pyproject.toml y src/
CONFIG    = SRC_DIR/"configs"/"retailer_region-bagging-ce-on-tf-0p1-le-1-eta-0p05-md-4-ss-0p6.toml"
GLOBAL_CSV = SRC_DIR/"results"/"resultados_globales"/"server_metrics.csv"
TRAIN_CSV  = SRC_DIR/"results"/"resultados_globales"/"train_metrics.csv"

# Borramos viejos
for p in (GLOBAL_CSV, TRAIN_CSV):
    if p.exists():
        print(f"✅ Borrado previo de {p}")
        p.unlink()

# Ejecutamos Flower y capturamos logs
print("\n Ejecutando experimento único…")
t0 = time.perf_counter()
res = subprocess.run(
    ["flwr", "run", ".", "--run-config", str(CONFIG)],
    cwd=str(SRC_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
dt = time.perf_counter() - t0

# Mostramos resultados
print(f"\n Duración: {dt:.1f}s  returncode={res.returncode}")
print("\n=== STDOUT ===")
print(res.stdout)
print("\n=== STDERR ===")
print(res.stderr)

# ¿Se generaron los CSV?
print("\nserver_metrics.csv existe?", GLOBAL_CSV.exists())
if GLOBAL_CSV.exists():
    dfg = pd.read_csv(GLOBAL_CSV)
    display(dfg)

print("\ntrain_metrics.csv existe?", TRAIN_CSV.exists())
if TRAIN_CSV.exists():
    dft = pd.read_csv(TRAIN_CSV)
    display(dft)



In [4]:
from pathlib import Path
import pandas as pd
import xgboost as xgb

# Ruta relativa desde este notebook:
models_dir = Path("../results/resultados_globales/models").resolve()
print("Buscando JSONs en:", models_dir)
print("¿Existe?", models_dir.exists())

model_paths = sorted(models_dir.glob("*_round_*.json"))
print(f"Encontrados {len(model_paths)} modelos:")
for p in model_paths:
    print("  ", p.name)

rows = []
for path in model_paths:
    rnd = int(path.stem.split("_round_")[-1])
    bst = xgb.Booster()
    bst.load_model(str(path))
    imp = bst.get_score(importance_type="gain")
    df = (
        pd.DataFrame.from_dict(imp, orient="index", columns=["gain"])
          .reset_index()
          .rename(columns={"index":"feature"})
    )
    df["round"] = rnd
    rows.append(df)

if not rows:
    raise RuntimeError(f"No hay datos para concatenar en {models_dir}")

full_imp = pd.concat(rows, ignore_index=True)
out_csv = Path("../results/resultados_globales/feature_importances.csv").resolve()
full_imp.to_csv(out_csv, index=False)
print("CSV generado en:", out_csv, "con", full_imp.shape[0], "filas")




Buscando JSONs en: C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\results\resultados_globales\models
¿Existe? True
Encontrados 3 modelos:
   retailer_region-bagging-ce-on-tf-0p1-le-1-eta-0p05-md-4-ss-0p6_2025-06-24_round_1.json
   retailer_region-bagging-ce-on-tf-0p1-le-1-eta-0p05-md-4-ss-0p6_2025-06-24_round_2.json
   retailer_region-bagging-ce-on-tf-0p1-le-1-eta-0p05-md-4-ss-0p6_2025-06-24_round_3.json
CSV generado en: C:\Users\PC\Desktop\s.o.e\Estudios\U-4\Segundo Cuatri\TFG\Codigo-Aplicacion de Aprendizaje Federado\src\results\resultados_globales\feature_importances.csv con 30 filas


### Conjunto de experimentos

In [ ]:
import subprocess, time, sys
from pathlib import Path

# Add the parent directory to sys.path so local modules can be imported
sys.path.append(str(Path("..").resolve()))

import metrics

# define rutas base
SRC_DIR    = Path("..")            # src/, donde está pyproject.toml
CONFIG_DIR = SRC_DIR / "configs"   # src/configs
RESULTS_DIR= SRC_DIR / "results"   # src/results


In [ ]:
import subprocess, time, toml
import pandas as pd

FLWR_CMD = ["flwr", "run", ".", "--run-config"]

def run_batch(pattern="*.toml"):
    cfg_paths = sorted(CONFIG_DIR.glob(pattern))
    print(f"Se lanzarán {len(cfg_paths)} experimentos…\n")

    for i, cfg in enumerate(cfg_paths, 1):
        cfg_abs = cfg.resolve()
        cfg_dict = toml.loads(cfg_abs.read_text(encoding="utf-8"))
        run_id   = cfg_dict["run-id"]

        # Prepara CSV de métricas para ESTE experimento
        exp_csv = RESULTS_DIR / f"{cfg_abs.stem}.csv"
        metrics.init_metrics_csv(exp_csv)

        print(f"({i}/{len(cfg_paths)}) 🚀  {cfg_abs.name}")
        t0 = time.perf_counter()

        # Arranca Flower (server + clients) dentro de src/
        proc = subprocess.run(
            FLWR_CMD + [str(cfg_abs)],
            cwd=str(SRC_DIR),
            text=True
        )

        dt = time.perf_counter() - t0
        print(f"⏱️  {dt:.1f}s — returncode: {proc.returncode}")

        if proc.returncode != 0:
            raise RuntimeError(f"{cfg_abs.name} terminó con error")

        # Lee las métricas globales que creó Flower
        server_csv = RESULTS_DIR / "server_metrics.csv"
        df = pd.read_csv(server_csv)

        # Filtra solo las rondas de ESTE experimento
        exp_df = df[df["experiment"] == run_id]
        if exp_df.empty:
            print("⚠️ No se encontraron métricas globales para", run_id)
        else:
            # Apendea cada ronda al CSV individual
            for _, row in exp_df.iterrows():
                metrics.append_round(
                    exp_csv,
                    {"config": cfg_abs.stem},
                    int(row["round"]),
                    float(row["rmse"]),
                    dt,                  # puedes medir tempo more fino si quieres
                )
            print(f"✅ Métricas guardadas en {exp_csv}")

    print("\n🎉 Todos los experimentos han terminado correctamente.")


In [ ]:
# O lanza todo:
run_batch()
